# 02. Baseline Model

This notebook is about setting an honest starting point.

Before trying stronger models, I want to know how far I can get with the simplest reasonable setup. That gives me a benchmark I can trust later.

So the question here is very simple:

**How well can this project do before things get fancy?**

To answer that, I compare a naive mean predictor with a basic linear regression model and keep the preprocessing as light as possible.


## Modeling Setup
### Target variable
We predict **`avg_ctc`**, which represents the midpoint of the disclosed salary range.

### Features used
We use only a small, interpretable feature set:
- `min_exp`
- `max_exp`
- `job_title`
- `location`

### Evaluation metric
We use **Mean Absolute Error (MAE)** as the primary metric.

Why MAE?
- Salary prediction is naturally measured in currency units.
- MAE is easy to interpret: it tells us the average absolute rupee error.
- It is more business-readable than squared-error metrics.

We also report **R?** as a secondary metric to show how much variance the baseline explains.


In [ ]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


## Load Data
We use the train and test splits created in the previous notebook.
To keep the baseline honest and clean, we train only on rows where the target salary is available.


In [ ]:
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

train = train[train['avg_ctc'].notna()].copy()
test = test[test['avg_ctc'].notna()].copy()

print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('Missing target in train:', train['avg_ctc'].isna().sum())
print('Missing target in test:', test['avg_ctc'].isna().sum())


## Define Target and Features
This baseline keeps the feature set intentionally small.

We avoid complex feature engineering, interaction terms, and model-specific transformations. That helps us understand what the simplest structured model can already capture.


In [ ]:
target = 'avg_ctc'
features = ['min_exp', 'max_exp', 'job_title', 'location']

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print('Target:', target)
print('Features:', features)
X_train.head()


## Minimal Preprocessing
The preprocessing is deliberately light but correct:
- numeric columns: median imputation
- categorical columns: most-frequent imputation + one-hot encoding

This is enough for a clean linear baseline without introducing unnecessary complexity.


In [ ]:
numeric_features = ['min_exp', 'max_exp']
categorical_features = ['job_title', 'location']

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
            ]),
            numeric_features,
        ),
        (
            'cat',
            Pipeline([
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('onehot', OneHotEncoder(handle_unknown='ignore')),
            ]),
            categorical_features,
        ),
    ]
)

preprocessor


## Baseline Models
We compare two simple baselines:

1. **Mean baseline**
   Predict the same average salary for every row.

2. **Linear Regression baseline**
   Learn a simple linear relationship between salary and the selected features.

If Linear Regression cannot beat the mean baseline by a useful margin, then the feature set or data quality may not support modeling well.


In [ ]:
models = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Linear regression baseline': LinearRegression(),
}


In [ ]:
results = []
fitted_models = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocess', preprocessor),
        ('model', model),
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    fitted_models[name] = pipeline
    results.append({
        'model': name,
        'mae': mean_absolute_error(y_test, predictions),
        'r2': r2_score(y_test, predictions),
    })

results_df = pd.DataFrame(results).sort_values('mae').reset_index(drop=True)
results_df


## Baseline Comparison
This table tells us whether a simple structured model is already useful.

The mean baseline gives us the minimum benchmark any predictive model should beat. The linear regression baseline tests whether basic relationships in experience, role, and location are already informative.


In [ ]:
results_df.assign(mae=results_df['mae'].round(2), r2=results_df['r2'].round(4))


In [ ]:
best_model = fitted_models['Linear regression baseline']
linear_preds = best_model.predict(X_test)
comparison = pd.DataFrame({
    'actual_salary': y_test.values,
    'predicted_salary': linear_preds,
    'absolute_error': abs(y_test.values - linear_preds),
})
comparison.head(10)


## Interpretation
### What the results mean
- The **mean baseline** shows what happens when we ignore all input information.
- The **linear regression baseline** tests whether a very simple model can extract signal from the chosen features.

If Linear Regression reduces MAE substantially relative to the mean baseline, that means the dataset contains usable predictive structure even before moving to more advanced models.


In [ ]:
mae_mean = results_df.loc[results_df['model'] == 'Mean baseline', 'mae'].iloc[0]
mae_lr = results_df.loc[results_df['model'] == 'Linear regression baseline', 'mae'].iloc[0]
mae_improvement = mae_mean - mae_lr
mae_improvement_pct = (mae_improvement / mae_mean) * 100

summary = pd.DataFrame({
    'metric': ['Mean baseline MAE', 'Linear regression MAE', 'Absolute MAE improvement', 'Relative MAE improvement %'],
    'value': [mae_mean, mae_lr, mae_improvement, mae_improvement_pct],
})
summary.assign(value=summary['value'].round(2))


## Baseline Takeaways
The baseline does its job well here.

A simple linear model beats the naive mean predictor by a clear margin, which tells me the features already carry real salary signal. That is exactly what I want from a baseline: not perfection, just a believable starting point.

It also gives the next notebook a clear challenge. Any stronger model should beat this result for the improvement to feel real.
